# Notebook 3 – Evaluate Models & Select the Best

**Purpose:** Load all trained models from S3, evaluate them on the held-out test set, compare performance with confusion matrices and classification reports, then save the best model with a dedicated tag in S3.

**Run this AFTER `02_train_models.ipynb`.**

---
### What this notebook does
1. Loads config and all model weights from S3
2. Evaluates each model on the test set
3. Generates per-class precision / recall / F1 reports
4. Draws confusion matrices for every model
5. Selects the best model (highest test accuracy)
6. Uploads the best model to a dedicated S3 key (`best_model/`) and saves a `results_summary.json`

## Cell 1 – Imports & Configuration

In [ ]:
import os
import json
import warnings
warnings.filterwarnings("ignore")

import boto3
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
)
from tqdm.notebook import tqdm

# ── Load project config ───────────────────────────────────────────────────────
CONFIG_FILE = "config.json"
if not os.path.exists(CONFIG_FILE):
    raise FileNotFoundError(
        f"'{CONFIG_FILE}' not found. Please run 01_setup_and_data.ipynb first."
    )

with open(CONFIG_FILE) as f:
    config = json.load(f)

S3_BUCKET        = config["S3_BUCKET"]
S3_DATA_PREFIX   = config["S3_DATA_PREFIX"]
S3_MODEL_PREFIX  = config["S3_MODEL_PREFIX"]
NUM_CLASSES      = config["NUM_CLASSES"]
CLASS_NAMES      = config["CLASS_NAMES"]
S3_MODEL_URIS    = config.get("s3_model_uris", {})
LOCAL_DATA_DIR   = "/tmp/banana_dataset"
LOCAL_MODELS_DIR = config.get("local_models_dir", "/tmp/trained_models")
EVAL_DIR         = "/tmp/eval_outputs"

INPUT_SIZE  = 224
BATCH_SIZE  = 16
NUM_WORKERS = 4

DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

os.makedirs(EVAL_DIR, exist_ok=True)

print("=" * 55)
print("Evaluation Configuration")
print("=" * 55)
print(f"  S3 Bucket     : {S3_BUCKET}")
print(f"  Classes ({NUM_CLASSES})   : {CLASS_NAMES}")
print(f"  Device        : {DEVICE}")
print(f"  Models found in config: {list(S3_MODEL_URIS.keys()) or 'None – will scan local dir'}")

## Cell 2 – Load Model Weights from S3

In [ ]:
def build_model(model_name: str, num_classes: int) -> nn.Module:
    """Reconstruct a model architecture (mirrors the builder in Notebook 2)."""
    if model_name == "MobileNetV3":
        model = models.mobilenet_v3_large(weights=None)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
    elif model_name == "EfficientNetB0":
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    elif model_name == "ResNet50":
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return model


def download_model_from_s3(s3_uri: str, local_dir: str) -> str:
    """
    Download a .pth file from s3://bucket/key to local_dir.
    Returns the local file path.
    """
    # Parse URI
    without_scheme = s3_uri[len("s3://"):]
    bucket, key = without_scheme.split("/", 1)
    filename = os.path.basename(key)
    local_path = os.path.join(local_dir, filename)

    if os.path.exists(local_path):
        print(f"  (cached) {local_path}")
        return local_path

    os.makedirs(local_dir, exist_ok=True)
    try:
        boto3.client("s3").download_file(bucket, key, local_path)
        print(f"  ⬇️  Downloaded {s3_uri} → {local_path}")
        return local_path
    except Exception as e:
        raise RuntimeError(f"Could not download {s3_uri}: {e}") from e


def load_trained_model(model_name: str, local_path: str, num_classes: int, device: torch.device) -> nn.Module:
    """Build the architecture and load weights from a .pth file."""
    model = build_model(model_name, num_classes)
    try:
        state = torch.load(local_path, map_location=device)
        model.load_state_dict(state)
    except Exception as e:
        raise RuntimeError(f"Failed to load weights from {local_path}: {e}") from e
    model = model.to(device)
    model.eval()
    return model


# Determine local paths: prefer S3 URIs from config, fall back to local dir scan
print("Loading model weights ...\n")
loaded_models = {}

model_local_paths = {}

if S3_MODEL_URIS:
    for model_name, uri in S3_MODEL_URIS.items():
        try:
            lp = download_model_from_s3(uri, LOCAL_MODELS_DIR)
            model_local_paths[model_name] = lp
        except RuntimeError as e:
            print(f"  ❌ Could not download {model_name}: {e}")
else:
    # Fall back to scanning local models dir
    print(f"  No S3 URIs in config – scanning {LOCAL_MODELS_DIR} ...")
    if not os.path.isdir(LOCAL_MODELS_DIR):
        raise FileNotFoundError(
            f"{LOCAL_MODELS_DIR} does not exist. "
            "Run 02_train_models.ipynb first."
        )
    for fname in os.listdir(LOCAL_MODELS_DIR):
        if fname.endswith(".pth"):
            name = fname.replace("_banana_ripeness.pth", "")
            model_local_paths[name] = os.path.join(LOCAL_MODELS_DIR, fname)

if not model_local_paths:
    raise RuntimeError(
        "No model weight files found. "
        "Please run 02_train_models.ipynb first."
    )

for model_name, lp in model_local_paths.items():
    try:
        loaded_models[model_name] = load_trained_model(model_name, lp, NUM_CLASSES, DEVICE)
        print(f"  ✅ {model_name} loaded from {lp}")
    except Exception as e:
        print(f"  ❌ Failed to load {model_name}: {e}")

## Cell 3 – Prepare Test DataLoader

In [ ]:
def find_split_dir(root: str, split: str) -> str:
    """Locate a split folder, checking one level of nesting."""
    direct = os.path.join(root, split)
    if os.path.isdir(direct):
        return direct
    for sub in os.listdir(root):
        candidate = os.path.join(root, sub, split)
        if os.path.isdir(candidate):
            return candidate
    raise FileNotFoundError(f"Could not find '{split}' folder under {root}.")


val_test_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

try:
    test_dir = find_split_dir(LOCAL_DATA_DIR, "test")
    test_dataset = datasets.ImageFolder(test_dir, val_test_transform)
    test_loader  = DataLoader(
        test_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )
    print(f"✅ Test set: {len(test_dataset)} images across {len(test_dataset.classes)} classes")
    print(f"   Classes: {test_dataset.classes}")
except FileNotFoundError as e:
    print(f"❌ {e}")
    print("   Downloading dataset from S3 ...")
    import boto3
    import sagemaker
    session = sagemaker.Session()
    # Re-use same paginator download from Notebook 2
    s3 = boto3.client("s3")
    paginator = s3.get_paginator("list_objects_v2")
    pages = paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_DATA_PREFIX + "/")
    for page in pages:
        for obj in page.get("Contents", []):
            key = obj["Key"]
            relative = key[len(S3_DATA_PREFIX):].lstrip("/")
            lp = os.path.join(LOCAL_DATA_DIR, relative)
            os.makedirs(os.path.dirname(lp), exist_ok=True)
            if not os.path.exists(lp):
                s3.download_file(S3_BUCKET, key, lp)
    test_dir = find_split_dir(LOCAL_DATA_DIR, "test")
    test_dataset = datasets.ImageFolder(test_dir, val_test_transform)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    print(f"✅ Test set ready: {len(test_dataset)} images")

## Cell 4 – Evaluate All Models

In [ ]:
def evaluate_model(model: nn.Module, loader: DataLoader, device: torch.device):
    """
    Run inference on the DataLoader and return
    (all_labels, all_preds, accuracy).
    """
    model.eval()
    all_labels = []
    all_preds  = []

    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc="Evaluating", leave=False):
            inputs = inputs.to(device, non_blocking=True)
            try:
                outputs = model(inputs)
            except RuntimeError as e:
                print(f"  ⚠️  Inference error: {e}")
                raise
            _, preds = torch.max(outputs, 1)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    return np.array(all_labels), np.array(all_preds), accuracy


eval_results = {}   # model_name -> {labels, preds, accuracy, report, cm}

for model_name, model in loaded_models.items():
    print(f"\nEvaluating {model_name} on test set ...")
    try:
        labels, preds, acc = evaluate_model(model, test_loader, DEVICE)
        report = classification_report(
            labels, preds,
            target_names=CLASS_NAMES,
            output_dict=True,
        )
        cm = confusion_matrix(labels, preds)

        eval_results[model_name] = {
            "labels":   labels,
            "preds":    preds,
            "accuracy": acc,
            "report":   report,
            "cm":       cm,
        }
        print(f"  Test Accuracy: {acc:.4f}")
    except Exception as e:
        print(f"  ❌ Evaluation failed for {model_name}: {e}")
        eval_results[model_name] = None

## Cell 5 – Classification Reports

In [ ]:
for model_name, res in eval_results.items():
    if res is None:
        print(f"\n{model_name}: evaluation failed, skipping.")
        continue
    print(f"\n{'=' * 55}")
    print(f"  {model_name} – Test Accuracy: {res['accuracy']:.4f}")
    print(f"{'=' * 55}")
    print(classification_report(
        res["labels"], res["preds"], target_names=CLASS_NAMES
    ))

## Cell 6 – Confusion Matrices

In [ ]:
successful_evals = {k: v for k, v in eval_results.items() if v is not None}

if not successful_evals:
    print("No successful evaluations to plot.")
else:
    n_models = len(successful_evals)
    fig, axes = plt.subplots(1, n_models, figsize=(7 * n_models, 6))
    if n_models == 1:
        axes = [axes]

    for ax, (model_name, res) in zip(axes, successful_evals.items()):
        cm_norm = res["cm"].astype(float) / res["cm"].sum(axis=1, keepdims=True)
        sns.heatmap(
            cm_norm,
            annot=True,
            fmt=".2f",
            xticklabels=CLASS_NAMES,
            yticklabels=CLASS_NAMES,
            cmap="Blues",
            ax=ax,
        )
        ax.set_title(f"{model_name}\nAcc = {res['accuracy']:.4f}", fontsize=11)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        plt.setp(ax.get_xticklabels(), rotation=45, ha="right")

    plt.tight_layout()
    cm_path = os.path.join(EVAL_DIR, "confusion_matrices.png")
    plt.savefig(cm_path, dpi=150)
    plt.show()
    print(f"\n✅ Confusion matrices saved to {cm_path}")

## Cell 7 – Select the Best Model

In [ ]:
if not successful_evals:
    raise RuntimeError("No models were successfully evaluated. Cannot select best.")

# Rank by test accuracy
ranked = sorted(
    successful_evals.items(),
    key=lambda kv: kv[1]["accuracy"],
    reverse=True,
)

print("\n" + "=" * 55)
print("Model Comparison (ranked by test accuracy)")
print("=" * 55)
for rank, (name, res) in enumerate(ranked, 1):
    marker = "🥇" if rank == 1 else ("🥈" if rank == 2 else "🥉")
    print(f"  {marker} #{rank} {name:15s}  Accuracy = {res['accuracy']:.4f}")

best_name, best_res = ranked[0]
print(f"\n🏆 Best model: {best_name} (test accuracy = {best_res['accuracy']:.4f})")

## Cell 8 – Upload Best Model to S3 with a Dedicated Key

In [ ]:
# Find the local .pth of the best model
best_local_path = os.path.join(
    LOCAL_MODELS_DIR, f"{best_name}_banana_ripeness.pth"
)

if not os.path.exists(best_local_path):
    # Try to find it by scanning the directory
    candidates = [
        f for f in os.listdir(LOCAL_MODELS_DIR)
        if best_name in f and f.endswith(".pth")
    ]
    if candidates:
        best_local_path = os.path.join(LOCAL_MODELS_DIR, candidates[0])
    else:
        raise FileNotFoundError(
            f"Could not find .pth for {best_name} in {LOCAL_MODELS_DIR}. "
            "Make sure Notebook 2 completed successfully."
        )

s3 = boto3.client("s3")
best_s3_key = f"{S3_MODEL_PREFIX}/best_model/{best_name}_best.pth"

try:
    s3.upload_file(best_local_path, S3_BUCKET, best_s3_key)
    best_s3_uri = f"s3://{S3_BUCKET}/{best_s3_key}"
    print(f"✅ Best model uploaded to: {best_s3_uri}")
except Exception as e:
    print(f"❌ Failed to upload best model: {e}")
    raise

# Also upload confusion matrix image
cm_s3_key = f"{S3_MODEL_PREFIX}/eval_artifacts/confusion_matrices.png"
cm_local  = os.path.join(EVAL_DIR, "confusion_matrices.png")
try:
    if os.path.exists(cm_local):
        s3.upload_file(cm_local, S3_BUCKET, cm_s3_key)
        print(f"✅ Confusion matrix uploaded to: s3://{S3_BUCKET}/{cm_s3_key}")
except Exception as e:
    print(f"⚠️  Could not upload confusion matrix: {e}")

## Cell 9 – Save Results Summary to JSON & S3

In [ ]:
summary = {
    "best_model": best_name,
    "best_model_test_accuracy": round(best_res["accuracy"], 6),
    "best_model_s3_uri": best_s3_uri,
    "all_results": {
        name: {
            "test_accuracy": round(res["accuracy"], 6),
            "precision_macro": round(res["report"]["macro avg"]["precision"], 6),
            "recall_macro":    round(res["report"]["macro avg"]["recall"],    6),
            "f1_macro":        round(res["report"]["macro avg"]["f1-score"],  6),
        }
        for name, res in successful_evals.items()
    },
    "class_names": CLASS_NAMES,
}

summary_local = os.path.join(EVAL_DIR, "results_summary.json")
with open(summary_local, "w") as f:
    json.dump(summary, f, indent=2)
print(f"✅ Results summary saved locally: {summary_local}")
print(json.dumps(summary, indent=2))

# Upload summary to S3
summary_s3_key = f"{S3_MODEL_PREFIX}/eval_artifacts/results_summary.json"
try:
    s3.upload_file(summary_local, S3_BUCKET, summary_s3_key)
    print(f"\n✅ Summary uploaded to: s3://{S3_BUCKET}/{summary_s3_key}")
except Exception as e:
    print(f"⚠️  Could not upload summary to S3: {e}")

# Update config.json with evaluation results
config["best_model"]            = best_name
config["best_model_test_acc"]   = round(best_res["accuracy"], 6)
config["best_model_s3_uri"]     = best_s3_uri
config["results_summary_s3"]    = f"s3://{S3_BUCKET}/{summary_s3_key}"
with open(CONFIG_FILE, "w") as f:
    json.dump(config, f, indent=2)
print(f"✅ config.json updated.")

## Cell 10 – Per-Class Accuracy Bar Chart

In [ ]:
if successful_evals:
    x = np.arange(len(CLASS_NAMES))
    width = 0.8 / len(successful_evals)

    fig, ax = plt.subplots(figsize=(12, 5))

    for i, (model_name, res) in enumerate(successful_evals.items()):
        per_class_acc = res["cm"].diagonal() / res["cm"].sum(axis=1)
        offset = (i - len(successful_evals) / 2 + 0.5) * width
        ax.bar(x + offset, per_class_acc, width, label=model_name)

    ax.set_xticks(x)
    ax.set_xticklabels(CLASS_NAMES, rotation=30, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("Per-Class Accuracy")
    ax.set_title("Per-Class Accuracy Comparison")
    ax.legend()
    ax.grid(axis="y", linestyle="--", alpha=0.7)

    plt.tight_layout()
    bar_path = os.path.join(EVAL_DIR, "per_class_accuracy.png")
    plt.savefig(bar_path, dpi=150)
    plt.show()
    print(f"✅ Per-class accuracy chart saved to {bar_path}")
    
    # Upload
    try:
        s3.upload_file(bar_path, S3_BUCKET, f"{S3_MODEL_PREFIX}/eval_artifacts/per_class_accuracy.png")
    except Exception:
        pass

---
## 🎉 All Done!

| Step | Where to find it |
|------|------------------|
| All trained models | `s3://<bucket>/banana-ripeness-models/` |
| **Best model** | `s3://<bucket>/banana-ripeness-models/best_model/` |
| Confusion matrices | `s3://<bucket>/banana-ripeness-models/eval_artifacts/confusion_matrices.png` |
| Per-class accuracy | `s3://<bucket>/banana-ripeness-models/eval_artifacts/per_class_accuracy.png` |
| Results summary | `s3://<bucket>/banana-ripeness-models/eval_artifacts/results_summary.json` |

> The bucket name is stored in `config.json` in the current directory.